In [1]:
import pandas as pd
import numpy as np
import os
User = os.environ.get('USERNAME')  # For Windows

# Replace 'file_path.csv' with the path to your CSV file
filepath='C:/Users/' + User + '/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv'
df = pd.read_csv(filepath)

PositionList =  [x for x in list(df['DuringWar']) if not (isinstance(x, float) and np.isnan(x)) and x != 'NaN']

In [2]:
# Extract position titles
positions = PositionList

from google.cloud import vision
from google.oauth2 import service_account
import os
def remove_file_extension(filename):
    root, _ = os.path.splitext(filename)
    return root

# Google Cloud Vision OCR function
def google_vision_ocr(image_path, service_account_file):
    # Setup credentials for GCV
    credentials = service_account.Credentials.from_service_account_file(service_account_file)
    client = vision.ImageAnnotatorClient(credentials=credentials)

    # Load the image
    with open(image_path, 'rb') as image_file:
        content = image_file.read()
    image = vision.Image(content=content)

    # Perform OCR using GCV
    response = client.text_detection(image=image)
    texts = response.text_annotations

    return texts

def extract_positions_from_gcv(texts, known_positions):
    extracted_positions = []
    for text in texts:
        for position in known_positions:
            if position in text.description:
                # Extract the x-coordinate of the left boundary of the bounding box
                x_min = min(vertex.x for vertex in text.bounding_poly.vertices)
                extracted_positions.append((x_min, position))
    return extracted_positions

def concatenate_and_find_positions(texts, known_positions):
    extracted_positions = []
    
    for position in known_positions:
        pos_len = len(position)
        for i in range(len(texts) - pos_len + 1):
            concatenated_text = ''.join([texts[j].description for j in range(i, i + pos_len)])
            if position in concatenated_text and texts[i].description.startswith(position[0]):
                x_min = texts[i].bounding_poly.vertices[0].x
                extracted_positions.append((x_min, concatenated_text))
    
    return extracted_positions

def standardize_your_ocr_positions(data, known_positions):
    standardized_positions = []
    for item in data:
        # Ensure that the item is a list and has at least 5 elements
        if isinstance(item, list) and len(item) >= 5:
            x_min, _, _, _, text = item
            for position in known_positions:
                # Check if the position is a substring of the text
                if position in text:
                    # Extract the position title and its x-coordinate
                    standardized_positions.append((x_min, position))
                    break
    return standardized_positions

# Function to extract text from your OCR data
def extract_text_from_ocr_data(ocr_data):
    return [item[-1] for sublist in ocr_data for item in sublist if isinstance(item, list) and item]

# Function to check if any position title is a substring of the text
def contains_position(text, positions):
    for position in positions:
        if position in text:
            return True, position
    return False, None


# Function to categorize entries by position based on the nearest title to the right
def categorize_by_position_to_the_right(data, merged_positions):
    # Sort merged positions by their x-coordinate
    merged_positions.sort(key=lambda x: x[0])

    # Create a dictionary to hold categorized data
    categorized_data = {position: [] for _, position in merged_positions}

    # Categorize each entry
    for entry in data:
        if isinstance(entry, list) and len(entry) >= 5:
            x_min, _, _, _, text = entry
            # Initialize variables to track the nearest position title to the right
            nearest_position_title = None
            nearest_position_distance = float('inf')

            # Find the nearest position title to the right
            for pos_x, pos_title in merged_positions:
                if pos_x > x_min and (pos_x - x_min) < nearest_position_distance:
                    nearest_position_title = pos_title
                    nearest_position_distance = pos_x - x_min

            # Assign the entry to the nearest position title
            if nearest_position_title:
                categorized_data[nearest_position_title].append(text)

    return categorized_data


def filter_positions_from_gcv(texts, known_positions):
    extracted_positions = []
    for text in texts:
        for position in known_positions:
            if position in text.description:
                extracted_positions.append(position)
    return extracted_positions

from PIL import Image

def adjust_gcv_coordinates(gcv_data, image_width):
    adjusted_data = []
    for x_min, position in gcv_data:
        # Adjust the x-coordinate to align with the other OCR system
        adjusted_x_min = image_width - x_min
        adjusted_data.append((adjusted_x_min, position))
    return adjusted_data

def group_entries_into_columns(categorized_entries, ocr_data, column_threshold=50):
    def build_text_to_coord(ocr_data):
        text_to_coord = {}
        for entry in ocr_data:
            # Ensure the entry has 5 elements
            if len(entry) == 5:
                x_min, y_min, x_max, y_max, text = entry
                text_to_coord[text] = (x_min, y_min, x_max, y_max)
        return text_to_coord

    text_to_coord = build_text_to_coord(ocr_data)

    columns_by_position = {position: {} for position in categorized_entries}

    def find_or_create_column(x_coord, columns):
        for col_x in columns:
            if abs(col_x - x_coord) <= column_threshold:
                return columns[col_x]
        columns[x_coord] = []
        return columns[x_coord]

    for position, entries in categorized_entries.items():
        for entry in entries:
            if entry in text_to_coord:
                x_coord, _, _, _ = text_to_coord[entry]
                column = find_or_create_column(x_coord, columns_by_position[position])
                column.append(entry)

    # Calculate bounding boxes and sort each column
    for position, columns in columns_by_position.items():
        for col_x in columns:
            bbox_entries = [text_to_coord[entry] for entry in columns[col_x] if entry in text_to_coord and len(text_to_coord[entry]) == 4]
            if bbox_entries:
                min_x = min(entry[0] for entry in bbox_entries)
                min_y = min(entry[1] for entry in bbox_entries)
                max_x = max(entry[2] for entry in bbox_entries)
                max_y = max(entry[3] for entry in bbox_entries)
                bbox = [[min_x, min_y], [max_x, max_y]]
            else:
                bbox = [[0, 0], [0, 0]]  # Placeholder if no entries in column

            columns[col_x] = {"entries": columns[col_x], "bbox": bbox}

    return columns_by_position

In [18]:
import os
import shutil

Year = 1941

# Paths
google_drive_path = 'G:/My Drive/Tokyo_Jobs/'+str(Year)
local_base_path = 'C:/Users/' + User + '/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)

if not os.path.exists(google_drive_path):
    os.makedirs(google_drive_path)

# Iterate through folders in Google Drive path
for folder_name in os.listdir(local_base_path):
    if folder_name.startswith("Page"):
        page_folder_path = os.path.join(google_drive_path, folder_name, "output")
        if os.path.isdir(page_folder_path):
            # Assuming only one subfolder exists inside the "output" folder
            output_subfolders = os.listdir(page_folder_path)
            if output_subfolders:
                random_folder_name = output_subfolders[0]
                json_folder_path = os.path.join(page_folder_path, random_folder_name, "json")
                
                if os.path.isdir(json_folder_path):
                    # Define local destination path
                    local_dest_path = os.path.join(local_base_path, folder_name, "NDLoutput")
                    os.makedirs(local_dest_path, exist_ok=True)

                    # Copy files from json folder to local destination
                    for file_name in os.listdir(json_folder_path):
                        source_file = os.path.join(json_folder_path, file_name)
                        dest_file = os.path.join(local_dest_path, file_name)
                        shutil.copy(source_file, dest_file)

print("Files copied successfully.")

Files copied successfully.


In [19]:
import cv2
import numpy as np
import json
import os
Year = 1941

def load_and_draw_bounding_boxes(image_path, ocr_data):
    # Load the image
    image = cv2.imread(image_path)

    # Draw each bounding box
    for entry in ocr_data[0]:  # Assuming the structure is a list of lists of lists
        x_min, y_min, x_max, y_max, text = entry
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(image, text, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    return image

def pad_to_same_width(image1, image2):
    # Determine the width difference
    delta_width = abs(image1.shape[1] - image2.shape[1])
    if image1 is None or image2 is None:
        return None, None
    if image1.shape[1] > image2.shape[1]:
        # Pad image2
        image2 = cv2.copyMakeBorder(image2, 0, 0, 0, delta_width, cv2.BORDER_CONSTANT)
    elif image1.shape[1] < image2.shape[1]:
        # Pad image1
        image1 = cv2.copyMakeBorder(image1, 0, 0, 0, delta_width, cv2.BORDER_CONSTANT)
    return image1, image2

def resize_for_display(image, max_height=800):
    """ Resize the image to fit on the screen for display purposes. """
    height, width = image.shape[:2]
    if height > max_height:
        scaling_factor = max_height / height
        new_width = int(width * scaling_factor)
        resized_image = cv2.resize(image, (new_width, max_height))
        return resized_image
    return image

# Paths
for Page in range(1,120,1):
    base_path = "C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/" +str(Year)+ "/Page"+f'{Page:03d}'
    ocr_files = {
        "left_bottom": "0001.json",
        "left_top": "0002.json",
        "right_bottom": "0003.json",
        "right_top": "0004.json"
    }

    quadrants = {}
    for position, ocr_file in ocr_files.items():
        image_path = os.path.join(base_path, 'Page',position + ".jpg")
        ocr_file_path = os.path.join(base_path, 'NDLoutput', ocr_file)

        # Load OCR results
        with open(ocr_file_path, 'r', encoding='utf-8') as file:
            ocr_data = json.load(file)

        # Load image and draw bounding boxes
        quadrants[position] = load_and_draw_bounding_boxes(image_path, ocr_data)

    # Reconstruct the full image
    # Pad images before stacking vertically
    try:
        left_top, left_bottom = pad_to_same_width(quadrants['left_top'], quadrants['left_bottom'])
        right_top, right_bottom = pad_to_same_width(quadrants['right_top'], quadrants['right_bottom'])

        # Vertically combine the top and bottom halves for each side
        left_side = np.vstack((left_top, left_bottom))
        right_side = np.vstack((right_top, right_bottom))

        # Horizontally combine the left and right sides
        full_image = np.hstack((left_side, right_side))

        # Resize the image for display
        resized_image = resize_for_display(full_image)

        # Save or display the reconstructed image
        #cv2.imwrite(os.path.join(base_path, "reconstructed_image.jpg"), full_image)
        cv2.imshow('Reconstructed Image', resized_image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

        # Define the filename for the saved image
        saved_image_filename = "reconstructed_image.jpg"

        # Full path for the saved image
        full_image_path = os.path.join(base_path, saved_image_filename)

        # Save the reconstructed image
        cv2.imwrite(full_image_path, full_image)
    except Exception as e:
        print(f"Error processing Page {Page}: {e}")
        continue  # Skip to the next page


Error processing Page 1: 'NoneType' object has no attribute 'shape'
Error processing Page 2: 'NoneType' object has no attribute 'shape'
Error processing Page 3: 'NoneType' object has no attribute 'shape'
Error processing Page 4: 'NoneType' object has no attribute 'shape'
Error processing Page 5: 'NoneType' object has no attribute 'shape'
Error processing Page 6: 'NoneType' object has no attribute 'shape'
Error processing Page 7: 'NoneType' object has no attribute 'shape'
Error processing Page 8: 'NoneType' object has no attribute 'shape'
Error processing Page 9: 'NoneType' object has no attribute 'shape'


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1941/Page010\\NDLoutput\\0001.json'

In [29]:
Page=1
base_path = "C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/" +str(Year)+ "/Page"+f'{Page:03d}'
ocr_files = {
    "left_bottom": "0001.json",
    "left_top": "0002.json",
    "right_bottom": "0003.json",
    "right_top": "0004.json"
}

quadrants = {}
for position, ocr_file in ocr_files.items():
    image_path = os.path.join(base_path, 'img',position + ".jpg")
    ocr_file_path = os.path.join(base_path, 'NDLoutput', ocr_file)

    # Load OCR results
    with open(ocr_file_path, 'r', encoding='utf-8') as file:
        ocr_data = json.load(file)

    # Load image and draw bounding boxes
    quadrants[position] = load_and_draw_bounding_boxes(image_path, ocr_data)


In [32]:
cv2.imshow('Reconstructed Image', quadrants['left_bottom'])
cv2.waitKey(0)


error: OpenCV(4.8.1) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1272: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'


In [36]:
import os
import re
Year=1941

def remove_file_extension(filename):
    root, _ = os.path.splitext(filename)
    return root
def open_json_in_folder(folder_path):
    json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    json_data = []

    for json_file in json_files:
        file_path = os.path.join(folder_path, json_file)
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            json_data.append(data)

    return json_data

def find_folder_with_format(base_path, format_regex):
    # Check each folder in the base directory
    for folder_name in os.listdir(base_path):
        # Check if the folder name matches the given format
        if re.match(format_regex, folder_name):
            # Construct the full path to the 'json' subfolder
            json_folder_path = os.path.join(base_path, folder_name, "json")
            return json_folder_path
    return None

# Example usage
base_path = 'G:/My Drive/Tokyo_Jobs/'+str(Year)+'/Page010/output'
format_regex = r'[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}'  # UUID format
folder_path = find_folder_with_format(base_path, format_regex)
os.listdir(folder_path)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'G:/My Drive/Tokyo_Jobs/1941/Page010/output\\73040438-9936-11ee-8f36-0242ac1c000c\\json'

In [33]:
Year=1941
Page=10
User = os.environ.get('USERNAME')  # For Windows
import json

# Path to your image and Google service account file
path='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/'
service_account_file = path+'Tokyo_Jobs/Environment/GoogleVision/practice-302516-01e0f7245b03.json'

image_folder = 'C:/Users/'+ User +'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1942/Page' + f'{Page:03d}' + '/Page/'
base_path = "G:/My Drive/Tokyo_Jobs/1942/Page010/output"

folder_path = find_folder_with_format(base_path, format_regex)
json_data_list = open_json_in_folder(folder_path)

for file in enumerate(os.listdir(image_folder)):
    filename = file[1]
    Index = remove_file_extension(filename)
    ocr_data = json_data_list[file[0]]
    with open(folder_path, 'r', encoding='utf-8') as file:
        json.dump(columns_by_position, file, ensure_ascii=False, indent=4)


    image_path = image_folder + filename

    # Load your image to get its width
    image = Image.open(image_path)

    # GCV OCR Results
    gcv_texts = google_vision_ocr(image_path, service_account_file)

    # Find positions in concatenated texts
    gcv_positions = concatenate_and_find_positions(gcv_texts[1:], positions)  # Skip the first element (entire text block)

    # Standardize format of position titles from your OCR data
    your_ocr_positions = standardize_your_ocr_positions(ocr_data, positions)

    # Merge GCV-detected positions with your original OCR positions
    merged_positions = list(set(gcv_positions + your_ocr_positions))

    # Now use merged_positions in the categorization process
    categorized_entries = categorize_by_position_to_the_right(ocr_data, merged_positions)


    # Display the results
    for position, names in categorized_entries.items():
        print(f"Position: {position}, Names: {names}")

    # Example usage
    column_threshold = 30  # Adjust as needed based on the layout of your document
    columns_by_position = group_entries_into_columns(categorized_entries, ocr_data, column_threshold)

    # Display the result
    for position, columns in columns_by_position.items():
        print(f"Position: {position}")
        for col_x, entries in columns.items():
            print(f"  Column starting at x={col_x}: {entries}")

    # Specify the file path where you want to save the JSON
    save_path = 'G:/My Drive/Tokyo_Jobs/1942/Page010' + '/NDLPosition'
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    file_path = save_path+'/'+Index+'.json'

    # Writing the dictionary to a JSON file
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(columns_by_position, file, ensure_ascii=False, indent=4)

    print(f"Data saved to {file_path}")

NameError: name 'find_folder_with_format' is not defined

- For Clerks

In [47]:
# Function to connect surnames and first names
def connect_surnames_firstnames(tokens):
    full_names = []
    i = 0
    while i < len(tokens) - 1:
        if '姓' in tokens[i]['part_of_speech'] and '名' in tokens[i+1]['part_of_speech']:
            full_name = tokens[i]['surface'] + tokens[i+1]['surface']
            full_names.append(full_name)
            i += 2  # Skip the next token since it's part of the current name
        else:
            i += 1
    return full_names

from janome.tokenizer import Tokenizer as janome_tokenizer
token_object = janome_tokenizer()
sentence = '浅野明司内藤政直木村幸信'

# Tokenize the sentence
tokens = token_object.tokenize(sentence)

# Convert tokens to a list of dictionaries
tokens_dict_list = []
for token in tokens:
    token_dict = {
        'surface': token.surface, 
        'part_of_speech': token.part_of_speech, 
        'base_form': token.base_form, 
        'reading': token.reading, 
        'phonetic': token.phonetic
    }
    tokens_dict_list.append(token_dict)

tokens_dict_list=[d for d in tokens_dict_list if '人名' in d['part_of_speech']]

# Extract full names
extracted_names = connect_surnames_firstnames(tokens_dict_list)
extracted_names


['浅野明司', '内藤政直', '木村幸信']

- For Workers

In [13]:
import re
def extract_wage_name(sentence, names):
    data = []
    for name in names:
        # Regex pattern to find an optional non-name string (wage) followed by the name
        pattern = r'([^\d]+)?' + re.escape(name)
        match = re.search(pattern, sentence)

        if match:
            # Extract the wage (if present) and the name
            wage = match.group(1).strip() if match.group(1) else ""
            data.append({"name": name, "wage": wage})
            # Remove the matched part from the sentence to avoid duplicate matching
            sentence = sentence.replace(match.group(), '')

    return json.dumps(data, ensure_ascii=False)

# Test the function with the provided sentence and names
sentence = '浅野明司内藤政直木村幸信'
names = ['浅野明司', '内藤政直', '木村幸信']
extracted_data = extract_wage_name(sentence, names)
extracted_data



'[{"name": "浅野明司", "wage": ""}, {"name": "内藤政直", "wage": ""}, {"name": "木村幸信", "wage": ""}]'

- For Managers

In [3]:
from janome.tokenizer import Tokenizer

def contains_name(text):
    tokenizer = Tokenizer()
    surname_found = False
    given_name_found = False

    for token in tokenizer.tokenize(text):
        part_of_speech = token.part_of_speech.split(',')[0]
        if part_of_speech == '姓':
            surname_found = True
        elif part_of_speech == '名':
            given_name_found = True

        if surname_found and given_name_found:
            return True

    return False

def categorize_japanese_text_with_names(text_list):
    categories = []
    wage_found = False

    for text in text_list:
        if "課長" in text or "掛長" in text:
            if contains_name(text):
                category = "position+name"
                wage_found = False  # Reset wage flag after finding position+name
            else:
                category = "position+name"  # If no name is found, categorize as address
                wage_found = False  # Reset wage flag after finding position+name
            
        elif not wage_found and any(char.isdigit() for char in text) or "一年" in text or "年" in text:
            category = "wage"
            wage_found = True  # Set wage flag to indicate wage has been found
        else:
            category = "address"
        categories.append({"text": text, "category": category})

    return categories

def modify_categorized_text(categorized_text):
    modified_text = []

    for i, entry in enumerate(categorized_text):
        if entry['category'] == 'position+name' and i > 0:
            # Modify the preceding entry to 'wage'
            previous_entry = categorized_text[i-1]
            previous_entry['category'] = 'wage'
            modified_text.append(previous_entry)
            modified_text.append(entry)
        elif entry['category'] != 'position+name':
            # Add non-'position+name' entries if they haven't been added already
            if not (modified_text and modified_text[-1] == entry):
                modified_text.append(entry)

    return modified_text


List=['□□', '統計資料掛長木間常理', '小石川白山伊豫一〇']
# Re-run the categorization with the new logic
categorized_texts_with_names = categorize_japanese_text_with_names(List)
modify_categorized_text(categorized_texts_with_names)

[{'text': '□□', 'category': 'wage'},
 {'text': '□□', 'category': 'wage'},
 {'text': '統計資料掛長木間常理', 'category': 'position+name'},
 {'text': '小石川白山伊豫一〇', 'category': 'address'}]

In [21]:
# Sample data (replace this with your actual data)
ocr_data =   [
    [
      1102,
      46,
      1136,
      111,
      "書主"
    ],
    [
      1076,
      46,
      1097,
      85,
      "三下"
    ],
    [
      1039,
      45,
      1060,
      86,
      "六上"
    ],
    [
      1000,
      45,
      1022,
      103,
      "月九五"
    ],
    [
      963,
      45,
      983,
      103,
      "月八五"
    ],
    [
      926,
      45,
      948,
      85,
      "七上"
    ],
    [
      1066,
      212,
      1094,
      385,
      "渡邊祥四郎"
    ],
    [
      1029,
      210,
      1057,
      387,
      "阿部貞道"
    ],
    [
      993,
      209,
      1020,
      383,
      "山根勝寛"
    ],
    [
      955,
      210,
      983,
      381,
      "西川重次郎"
    ],
    [
      917,
      206,
      946,
      379,
      "蓬田吉五郎"
    ],
    [
      1074,
      456,
      1095,
      495,
      "五下"
    ],
    [
      1037,
      458,
      1057,
      499,
      "六上"
    ],
    [
      1000,
      456,
      1020,
      494,
      "六下"
    ],
    [
      962,
      455,
      984,
      516,
      "月八五"
    ],
    [
      926,
      456,
      946,
      496,
      "七下"
    ],
    [
      1066,
      624,
      1094,
      790,
      "佐々木亭"
    ],
    [
      1027,
      622,
      1057,
      795,
      "八卷善雄"
    ],
    [
      989,
      618,
      1020,
      794,
      "大野城"
    ],
    [
      954,
      623,
      983,
      793,
      "小池保"
    ],
    [
      917,
      622,
      945,
      790,
      "青木やなぎ"
    ],
    [
      877,
      46,
      910,
      148,
      "枝手六下"
    ],
    [
      877,
      619,
      908,
      792,
      "鈴木嘉幸"
    ],
    [
      836,
      25,
      872,
      677,
      "雇鬼頭恭而加藤彰細"
    ],
    [
      834,
      398,
      870,
      789,
      "加藤彰細川"
    ],
    [
      798,
      144,
      826,
      288,
      "浅野明司"
    ],
    [
      718,
      44,
      753,
      141,
      "屬記員"
    ],
    [
      757,
      146,
      789,
      289,
      "塚本久校"
    ],
    [
      791,
      391,
      826,
      790,
      "内藤政直木村幸信"
    ],
    [
      676,
      34,
      713,
      797,
      "市數普及事務託□一二〇"
    ],
    [
      645,
      581,
      668,
      787,
      "豊島雑司ヶ谷一ノ七"
    ],
    [
      610,
      49,
      650,
      800,
      "市政映章務演記池田一夫"
    ],
    [
      581,
      580,
      602,
      767,
      "流橋柏木一ノ一六八"
    ],
    [
      561,
      584,
      582,
      719,
      "海部一大六大な"
    ],
    [
      493,
      42,
      532,
      236,
      "○統計課"
    ],
    [
      429,
      44,
      464,
      103,
      "主事"
    ],
    [
      402,
      42,
      422,
      82,
      "八七"
    ],
    [
      334,
      39,
      359,
      134,
      "一年二一〇〇・"
    ],
    [
      119,
      42,
      300,
      79,
      "□□"
    ],
    [
      473,
      264,
      515,
      436,
      "同芝橋五五日"
    ],
    [
      513,
      269,
      549,
      577,
      "芝區芝公園第二十三号地"
    ],
    [
      386,
      405,
      423,
      784,
      "課長中西清太郎"
    ],
    [
      360,
      583,
      383,
      754,
      "瀬野川田端四五"
    ],
    [
      320,
      400,
      358,
      787,
      "人口統計掛長田村俊雄"
    ],
    [
      296,
      582,
      319,
      763,
      "王子稲村五ノ九六一"
    ],
    [
      257,
      397,
      292,
      788,
      "勞務統計掛長倉田勝"
    ],
    [
      232,
      581,
      255,
      774,
      "本郷駒込神明三六四"
    ],
    [
      190,
      394,
      231,
      788,
      "消費統計掛長鈴木勝太郎"
    ],
    [
      167,
      578,
      191,
      786,
      "大舟馬込東二ノ一二八"
    ],
    [
      125,
      398,
      163,
      790,
      "統計資料掛長木間常理"
    ],
    [
      103,
      582,
      125,
      783,
      "小石川白山伊豫一〇"
    ],
    [
      83,
      583,
      105,
      733,
      "小結三九一"
    ]
  ]